# Tokenization - Step 1

Minimal first step: tokenize a sentence with a pretrained GPT-2 tokenizer, convert tokens to IDs, and decode back to text.

In [11]:
from transformers import AutoTokenizer

# Load a pre-trained tokenizer (e.g., GPT-2)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Input text
text = "Artificial intelligence is amazing!"

# Tokenize the text
tokens = tokenizer.tokenize(text)
print("Tokens:", tokens)

# Convert tokens to numerical IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print("Token IDs:", token_ids)

# Decode numerical IDs back to text
decoded_text = tokenizer.decode(token_ids)
print("Decoded Text:", decoded_text)

Tokens: ['Art', 'ificial', 'Ġintelligence', 'Ġis', 'Ġamazing', '!']
Token IDs: [8001, 9542, 4430, 318, 4998, 0]
Decoded Text: Artificial intelligence is amazing!


### Why This Output Looks Like This

`Tokens: ['Art', 'ificial', 'Ġintelligence', 'Ġis', 'Ġamazing', '!']`
- GPT-2 uses subword tokenization (Byte Pair Encoding), so words can be split into pieces.
- `Artificial` became `Art` + `ificial` because that split exists in the learned vocabulary.
- The `Ġ` prefix means "this token starts after a space".
  - So `Ġintelligence` means ` intelligence`.
  - Same for `Ġis` and `Ġamazing`.

`Token IDs: [8001, 9542, 4430, 318, 4998, 0]`
- Each token is mapped to an integer ID from GPT-2's vocabulary.
- These IDs are what the model actually processes.
- The same token always maps to the same ID for this tokenizer.

`Decoded Text: Artificial intelligence is amazing!`
- Decoding converts IDs back to text.
- The tokenizer rebuilds spaces from tokens with `Ġ` and merges subword pieces (`Art` + `ificial`).
- So the reconstructed sentence matches the original text.

## Step 2: Positional Encoding

Token embeddings alone do not store word order.

Positional encoding injects position information so the model can distinguish:
- `dog bites man`
- `man bites dog`

Below we implement sinusoidal positional encoding (the classic Transformer version).

In [12]:
import numpy as np

def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    angle_rads = pos * angle_rates

    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle_rads[:, 0::2])
    pe[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return pe

seq_len = 8
d_model = 16
pe = positional_encoding(seq_len, d_model)

print('Positional encoding shape:', pe.shape)
print('First 2 positions (first 8 dims):')
print(pe[:2, :8])

Positional encoding shape: (8, 16)
First 2 positions (first 8 dims):
[[0.         1.         0.         1.         0.         1.
  0.         1.        ]
 [0.84147098 0.54030231 0.31098359 0.95041528 0.09983342 0.99500417
  0.03161751 0.99950004]]


In [13]:
# Add positional encoding to token embeddings (toy demo)
np.random.seed(42)
token_embeddings = np.random.randn(seq_len, d_model)
input_with_position = token_embeddings + pe

print('Token embeddings shape:', token_embeddings.shape)
print('Input + position shape:', input_with_position.shape)
print('Example vector (position 0, first 6 dims):')
print(input_with_position[0, :6])

Token embeddings shape: (8, 16)
Input + position shape: (8, 16)
Example vector (position 0, first 6 dims):
[ 0.49671415  0.8617357   0.64768854  2.52302986 -0.23415337  0.76586304]


## Step 3: Attention (Scaled Dot-Product)

Attention answers: "Which other tokens should each token focus on?"

Core formula:
- `Attention(Q, K, V) = softmax(QK^T / sqrt(d_k))V`

We create tiny random `Q`, `K`, `V` to inspect attention weights.

In [14]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# Tiny self-attention demo (single head)
np.random.seed(7)
seq_len = 5
d_k = 8

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

scores = (Q @ K.T) / np.sqrt(d_k)
weights = softmax(scores, axis=-1)
attended = weights @ V

print('Attention scores shape:', scores.shape)
print('Attention weights shape:', weights.shape)
print('Attended output shape:', attended.shape)
print('\nAttention weights for token 0:')
print(np.round(weights[0], 3))

Attention scores shape: (5, 5)
Attention weights shape: (5, 5)
Attended output shape: (5, 8)

Attention weights for token 0:
[0.108 0.281 0.031 0.178 0.402]


## Step 4: FFN (Feed-Forward Network)

After attention, each token vector goes through the same small MLP (position-wise FFN):
- Dense -> ReLU -> Dense

This adds non-linearity and richer feature transformation.

In [15]:
import tensorflow as tf
from tensorflow.keras import layers

ffn = tf.keras.Sequential([
    layers.Dense(32, activation='relu'),
    layers.Dense(d_k)
])

attended_tf = tf.constant(attended, dtype=tf.float32)
ffn_out = ffn(attended_tf)

print('FFN input shape:', attended_tf.shape)
print('FFN output shape:', ffn_out.shape)
print('First token vector after FFN (first 6 dims):')
print(np.round(ffn_out.numpy()[0, :6], 3))

FFN input shape: (5, 8)
FFN output shape: (5, 8)
First token vector after FFN (first 6 dims):
[ 0.479  0.354  0.216  0.114 -0.556 -0.072]


## Step 5: The Full Picture (Mini Transformer Block Flow)

You now have the core path used in LLM blocks:
1. Tokens -> token embeddings
2. Add positional encoding
3. Self-attention
4. FFN

In full Transformer blocks, this also includes residual connections and layer normalization.